# Cross Subject — ERP-template pure ML, full retraining

Attach only `low-cost-motor-imagery-decoding-for-rehab-cross-subject` and run all cells in a fresh CPU kernel.
This notebook embeds its complete source and trains from raw competition CSVs.
It uses NumPy, SciPy, pandas, scikit-learn and joblib. No network access, neural
package, external dataset, cached feature file or pretrained checkpoint is required.

The frozen model uses exactly the `erp_pre` representation: eight EEG channels,
trial-local `[-2,0)` baseline scaling, a 20-Hz lowpass over `[0,2)`, 40-ms bins,
and clipping to [-20,20]. Each source-class mean waveform contributes eight
template channels. The two templates and each trial form 24-channel covariance
matrices with 10% isotropic shrinkage and trace normalization. A source-fitted
covariance reference, tangent representation, scaler and logistic classifier
(C=0.1) complete the model. Covariance removes each waveform's temporal mean.
The threshold is 0.5. Training is empirical, with no sample weights.

All 17 supplied source participants contribute to final fitting. Target people
contribute no fitted statistics or templates, and one target trial does not
affect another. This notebook performs no validation scoring or model selection.
It preserves the original linear baseline notebook as a separate candidate.
This is a mixed-evidence research alternative: its development advantage did
not improve the primary run-weighted confirmation score. Linear ERP remains
the project's default Cross Subject ML candidate; no settings were retuned
on confirmation outcomes.

Only Fz, C3, Cz, C4, PO7, Pz, PO8 and Oz enter the model. Markers provide trial
boundaries and source labels; timestamps check sampling continuity. Filenames
and trial order establish submission IDs. Native units and nominal 250-Hz
sampling are retained; the acquisition reference is unverified. The raw loader
checks `[-2,0)` baseline and `[0,4.8)` task support; the model uses `[0,2)`.
There is no target-batch adaptation or forced class count.

Outputs are `submission.csv`, `test_predictions.csv`, `metadata.csv`, a saved
`erp_template_model.joblib`, the exact embedded source and `manifest.json`.
The manifest records source/input/output hashes, build provenance, versions,
fit participants, runtime and exact immediate saved-model replay agreement.
The [official evaluation](https://www.kaggle.com/competitions/low-cost-motor-imagery-decoding-for-rehab-cross-subject/overview/evaluation)
specifies `ID,TARGET`, IDs 0 through 359, and lowercase `rest`/`move` labels.
Actual package versions are recorded; numerical results may differ across
BLAS or package versions. Full Kaggle execution remains necessary before an
eligible submission. No upload is performed.


In [ ]:
import os
from pathlib import Path

COMPETITION = 'cross_subject'
SLUG = 'low-cost-motor-imagery-decoding-for-rehab-cross-subject'
TEST_FOLDER = 'Cross participant test set'
EXPECTED_TEST_ROWS = 360
EXPECTED_TEST_FILES = 9
EXPECTED_TEST_PEOPLE = 3
NOTEBOOK_NAME = 'cross_subject_erp_template_ml.ipynb'
BUILD_ID = 'cbec5a7a8c01b63b8a537038c9caf02a810ec9625d6910abb57741fe6f0225cf'
BUILD_PROVENANCE = {'generator': 'scripts/build_erp_notebook.py', 'generator_sha256': '29e8888f60b4041b65a4205864a43852e56db07a7453ffeca03fdfbc73582df0', 'shared_builder_sha256': '5a8803f94c6cd9dcbd44452bff4dd22e6a45d6d11324236c0e1c06f813bfc0b9', 'shared_raw_loader_sha256': '3975c49dde268cecba259a6ab6639997a0713d3c960576d090fa1e533cf0f835', 'source_sha256': {'data.py': '40eebf639479ea6b34041b4c592c7e532b166e0db31b32973b3799df8b7e8c53', 'classical.py': '2709b2d4420edb5eb47781f03fa791925bbb3785ec646b4c503c95a8a5da22f8', 'erp_template.py': 'beb0f848ea519e8b9a850a458f2edb3e59c18fdb4a94987a211dea2583732462', 'submission.py': '8d68409c4cc695229ee999aeab792fdd17829dcb32cc66d9428e0d66c3310473', '__init__.py': 'bc5382830b7ed016c341caa511f10dba7d12944b8f35c0c358095decb3c477bf'}, 'competition': 'cross_subject', 'spec': {'slug': 'low-cost-motor-imagery-decoding-for-rehab-cross-subject', 'recipe': 'erp_pre', 'test_folder': 'Cross participant test set', 'test_rows': 360, 'test_files': 9, 'test_people': 3}, 'model': {'recipe': 'erp_pre', 'C': 0.1, 'shrink': 0.1, 'threshold': 0.5, 'weighting': 'empirical; no sample weights'}}
DATA_ROOT = os.environ.get('EEG_DATA_ROOT') or None
OUTPUT_DIR = Path(os.environ.get('EEG_OUTPUT_DIR', '/kaggle/working'))
RECIPE, C, SHRINK, SEED = 'erp_pre', .1, .1, 20261003
THREADS = int(os.environ.get('EEG_THREADS', '1'))
if THREADS < 1:
    raise ValueError('THREADS must be positive')
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[variable] = str(THREADS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any((OUTPUT_DIR / name).exists() for name in ['manifest.json', 'submission.csv',
                                               'test_predictions.csv', 'erp_template_model.joblib']):
    raise FileExistsError('Use a fresh output directory; existing candidate files are preserved')


## Embedded source and reproducible environment


In [ ]:
import hashlib
import importlib
import json
import platform
import random
import sys
import time
from dataclasses import asdict

STARTED = time.monotonic()
SOURCE_HASHES = {'data.py': '40eebf639479ea6b34041b4c592c7e532b166e0db31b32973b3799df8b7e8c53', 'classical.py': '2709b2d4420edb5eb47781f03fa791925bbb3785ec646b4c503c95a8a5da22f8', 'erp_template.py': 'beb0f848ea519e8b9a850a458f2edb3e59c18fdb4a94987a211dea2583732462', 'submission.py': '8d68409c4cc695229ee999aeab792fdd17829dcb32cc66d9428e0d66c3310473', '__init__.py': 'bc5382830b7ed016c341caa511f10dba7d12944b8f35c0c358095decb3c477bf'}
SOURCES = {'data.py': '"""Marker-led loading with an explicit native-sample epoch contract.\n\nOnly named EEG columns enter X. Time, marker spellings and file metadata are\naudit/segmentation information, never predictor columns. Each competition is\nread independently; no data matching across competitions is implemented.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nCHANNELS = ("Fz", "C3", "Cz", "C4", "PO7", "Pz", "PO8", "Oz")\nFS = 250\nCUE_INDEX = 750\nEPOCH_SAMPLES = 2000\nLABELS = {"rest": 0, "move": 1}\nCOMPETITIONS = ("within_subject", "cross_subject")\nSTART_MARKERS = {"trial_start", "trail_start"}\nKNOWN_MARKERS = START_MARKERS | {"rest", "move", "cue_start", "cue_stop", "exp_start", "exp_stop"}\n\n\n@dataclass(frozen=True)\nclass Trial:\n    trial_start: int\n    cue: int\n    stop: int\n    label: str\n\n\ndef parse_markers(markers, split: str) -> tuple[list[Trial], dict]:\n    """Pair each cue with its own fixation start and first subsequent cue_stop.\n\n    Extra stop markers outside a trial are reported, not converted into trials.\n    Missing boundaries, multiple cues, mixed labeled/test cues and unknown\n    marker types are rejected rather than silently losing or relabeling trials.\n    """\n    if split not in {"train", "test"}:\n        raise ValueError(f"Invalid split: {split}")\n    trials: list[Trial] = []\n    counts: Counter = Counter()\n    orphan_stops, unused_starts = [], []\n    start = cue = None\n    label = ""\n    for row, value in enumerate(markers):\n        if pd.isna(value):\n            continue\n        marker = str(value).strip().lower()\n        if not marker:\n            continue\n        counts[marker] += 1\n        if marker not in KNOWN_MARKERS:\n            raise ValueError(f"Unknown marker {marker!r} at row {row}")\n        if marker in START_MARKERS:\n            if cue is not None:\n                raise ValueError(f"Missing cue_stop before trial start at row {row}")\n            if start is not None:\n                unused_starts.append(start)\n            start = row\n        elif marker in {"rest", "move", "cue_start"}:\n            if (split == "train") != (marker in LABELS):\n                raise ValueError(f"Unexpected {marker} in {split} at row {row}")\n            if cue is not None:\n                raise ValueError(f"Multiple cues before cue_stop at row {row}")\n            if start is None:\n                raise ValueError(f"Cue without trial start at row {row}")\n            cue, label = row, marker if split == "train" else ""\n        elif marker == "cue_stop":\n            if cue is None:\n                orphan_stops.append(row)\n            else:\n                trials.append(Trial(start, cue, row, label))\n                start = cue = None\n        elif marker == "exp_stop":\n            if cue is not None:\n                raise ValueError(f"Experiment ended before cue_stop at row {row}")\n            if start is not None:\n                unused_starts.append(start)\n                start = None\n        elif marker == "exp_start" and (start is not None or cue is not None):\n            raise ValueError(f"Experiment restarted inside a trial at row {row}")\n    if cue is not None:\n        raise ValueError("File ended with an unclosed cue")\n    if start is not None:\n        unused_starts.append(start)\n    if not trials:\n        raise ValueError("No complete trials found")\n    return trials, {"marker_counts": dict(counts), "orphan_stop_rows": orphan_stops,\n                    "unused_start_rows": unused_starts}\n\n\ndef identify_file(path: Path, competition: str, split: str) -> tuple[str, int]:\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    patterns = {\n        "train": r"(S\\d{3})-(\\d{3})_eeg\\.csv",\n        "cross_subject": r"(S\\d{3})_test_(\\d{3})_eeg\\.csv",\n        "within_subject": r"(S\\d{3})_test\\.csv",\n    }\n    match = re.fullmatch(patterns["train" if split == "train" else competition], path.name)\n    if match is None:\n        raise ValueError(f"Unexpected EEG file name: {path.name}")\n    return match[1], int(match[2]) if len(match.groups()) == 2 else 3\n\n\ndef extract_epochs(frame: pd.DataFrame, trials: list[Trial]) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[dict], dict]:\n    """Extract [-750,1250) native rows around cue; no temporal resampling.\n\n    X contains real recorded samples outside the exact fixation/task bounds,\n    flagged separately by phase_valid_mask. Missing/nonfinite samples and\n    samples across a timestamp discontinuity from the cue are NaN, with\n    valid_mask=False. A discontinuity is <=0 or >1.5 * median positive dt.\n    Phase boundaries are half-open: baseline starts at trial_start and task\n    ends immediately before cue_stop. Full native phase lengths are retained.\n    """\n    required = {"time", "Marker_val", *CHANNELS}\n    missing = required - set(frame.columns)\n    if missing:\n        raise ValueError(f"Missing columns: {sorted(missing)}")\n    time = frame["time"].to_numpy(dtype=np.float64)\n    eeg = frame.loc[:, CHANNELS].to_numpy(dtype=np.float32)\n    if len(time) < 2 or not np.isfinite(time).all():\n        raise ValueError("Recording must have at least two finite timestamps")\n    dt = np.diff(time)\n    positive = dt[dt > 0]\n    if not len(positive):\n        raise ValueError("Recording timestamps never increase")\n    median_dt = float(np.median(positive))\n    if not 0.0035 < median_dt < 0.0045:\n        raise ValueError(f"Unexpected sampling interval: {median_dt}")\n    discontinuity = (dt <= 0) | (dt > 1.5 * median_dt)\n    segments = np.r_[0, np.cumsum(discontinuity)]\n    finite_rows = np.isfinite(eeg).all(axis=1)\n    X = np.full((len(trials), len(CHANNELS), EPOCH_SAMPLES), np.nan, dtype=np.float32)\n    valid = np.zeros((len(trials), EPOCH_SAMPLES), dtype=bool)\n    phase_valid = valid.copy()\n    metadata = []\n    offsets = np.arange(-CUE_INDEX, EPOCH_SAMPLES - CUE_INDEX)\n    for i, trial in enumerate(trials):\n        if not 0 <= trial.trial_start < trial.cue < trial.stop < len(frame):\n            raise ValueError(f"Invalid native trial boundaries: {trial}")\n        rows = trial.cue + offsets\n        inside = (rows >= 0) & (rows < len(frame))\n        idx = np.flatnonzero(inside)\n        valid[i, idx] = finite_rows[rows[idx]] & (segments[rows[idx]] == segments[trial.cue])\n        good = np.flatnonzero(valid[i])\n        X[i][:, good] = eeg[rows[good]].T\n        phase_valid[i] = valid[i] & (rows >= trial.trial_start) & (rows < trial.stop)\n        metadata.append({\n            "trial_index": i, "trial_start_row": trial.trial_start,\n            "cue_row": trial.cue, "cue_stop_row": trial.stop,\n            "trial_start_time": float(time[trial.trial_start]),\n            "cue_time": float(time[trial.cue]), "cue_stop_time": float(time[trial.stop]),\n            "label": trial.label, "y": LABELS.get(trial.label, -1),\n            "baseline_samples": trial.cue - trial.trial_start,\n            "native_cue_samples": trial.stop - trial.cue,\n            "baseline_seconds": float(time[trial.cue] - time[trial.trial_start]),\n            "cue_seconds": float(time[trial.stop] - time[trial.cue]),\n            "valid_samples": int(valid[i].sum()),\n            "phase_valid_samples": int(phase_valid[i].sum()),\n            "baseline_2s_valid": bool(phase_valid[i, CUE_INDEX - 500:CUE_INDEX].all()),\n            "task_4p8s_valid": bool(phase_valid[i, CUE_INDEX:CUE_INDEX + 1200].all()),\n            "task_5s_valid": bool(phase_valid[i, CUE_INDEX:].all()),\n        })\n    timing = {\n        "median_dt_seconds": median_dt, "effective_fs_hz": 1.0 / median_dt,\n        "dt_min_seconds": float(dt.min()), "dt_max_seconds": float(dt.max()),\n        "dt_std_seconds": float(dt.std()), "discontinuity_rows": (np.flatnonzero(discontinuity) + 1).tolist(),\n        "nonpositive_intervals": int((dt <= 0).sum()),\n        "gap_intervals": int((dt > 1.5 * median_dt).sum()),\n        "max_abs_dt_deviation_seconds": float(np.max(np.abs(dt - median_dt))),\n        "first_time": float(time[0]), "last_time": float(time[-1]),\n    }\n    return X, valid, phase_valid, metadata, timing\n\n\ndef channel_quality(frame: pd.DataFrame) -> dict:\n    """Descriptive quality statistics in file units; no artifact cleaning."""\n    result = {}\n    for channel in CHANNELS:\n        values = frame[channel].to_numpy(dtype=np.float64)\n        finite = values[np.isfinite(values)]\n        if not len(finite):\n            result[channel] = {"finite_fraction": 0.0}\n            continue\n        diff = np.diff(values)\n        equal = diff == 0\n        edges = np.diff(np.r_[False, equal, False].astype(int))\n        starts, stops = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)\n        q = np.quantile(finite, [.001, .01, .5, .99, .999])\n        result[channel] = {\n            "finite_fraction": float(len(finite) / len(values)),\n            "min": float(finite.min()), "max": float(finite.max()),\n            "q001": float(q[0]), "q01": float(q[1]), "median": float(q[2]),\n            "q99": float(q[3]), "q999": float(q[4]), "std": float(finite.std()),\n            "max_abs_step": float(np.nanmax(np.abs(diff))),\n            "equal_adjacent_fraction": float(equal.mean()),\n            "longest_constant_samples": int((stops - starts).max() + 1) if len(starts) else 1,\n            "fraction_at_min": float(np.mean(values == finite.min())),\n            "fraction_at_max": float(np.mean(values == finite.max())),\n        }\n    return result\n\n\ndef competition_paths(data_root: Path, competition: str) -> list[tuple[str, Path]]:\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    root = data_root / competition\n    test_folder = "Single subject test set" if competition == "within_subject" else "Cross participant test set"\n    paths = [(split, p) for split, folder in [("train", "Training set"), ("test", test_folder)]\n             for p in sorted((root / folder).glob("*.csv"))]\n    if {split for split, _ in paths} != {"train", "test"}:\n        raise FileNotFoundError(f"Missing train/test files under {root}")\n    return paths\n\n\ndef audit_competition(data_root: Path, output_root: Path, competition: str) -> dict:\n    """Read and write exactly one competition; dictionaries never span tasks."""\n    paths = competition_paths(data_root, competition)\n    arrays, masks, phase_masks, metadata, file_reports = [], [], [], [], []\n    file_hashes, epoch_hashes = {}, {}\n    duplicate_files, duplicate_epochs = [], []\n    test_order = 0\n    for split, path in paths:\n        subject, run = identify_file(path, competition, split)\n        frame = pd.read_csv(path)\n        trials, marker_report = parse_markers(frame["Marker_val"], split)\n        X, valid, phase_valid, rows, timing = extract_epochs(frame, trials)\n        digest = hashlib.sha256(path.read_bytes()).hexdigest()\n        if digest in file_hashes:\n            duplicate_files.append([file_hashes[digest], path.name])\n        file_hashes[digest] = path.name\n        for local_index, row in enumerate(rows):\n            row.update({"epoch_index": len(metadata), "competition": competition,\n                        "split": split, "subject": subject, "run": run,\n                        "file": path.relative_to(data_root / competition).as_posix(),\n                        "test_order": test_order if split == "test" else -1})\n            if split == "test":\n                test_order += 1\n            # Hash original float64 numeric values, before cache quantization.\n            # Compare recorded EEG only within this competition.\n            cue_rows = frame.iloc[trials[local_index].cue:trials[local_index].cue + 1000]\n            cue_digest = hashlib.sha256(cue_rows.loc[:, CHANNELS].to_numpy(dtype=np.float64).tobytes()).hexdigest()\n            reference = {"file": row["file"], "trial_index": row["trial_index"], "split": split}\n            if cue_digest in epoch_hashes:\n                duplicate_epochs.append([epoch_hashes[cue_digest], reference])\n            epoch_hashes[cue_digest] = reference\n            metadata.append(row)\n        file_reports.append({"file": path.relative_to(data_root / competition).as_posix(),\n                             "split": split, "subject": subject, "run": run,\n                             "rows": len(frame), "columns": frame.columns.tolist(), "sha256": digest,\n                             "n_trials": len(trials), "labels": dict(Counter(t.label for t in trials if t.label)),\n                             **marker_report, **timing, "channels": channel_quality(frame),\n                             "overlapping_native_trials": [i for i in range(1, len(trials))\n                                                           if trials[i].trial_start <= trials[i - 1].stop],\n                             "overlapping_cached_epochs": [i for i in range(1, len(trials))\n                                                            if trials[i].cue - trials[i - 1].cue < EPOCH_SAMPLES]})\n        arrays.append(X); masks.append(valid); phase_masks.append(phase_valid)\n        print(f"{competition} {path.name}: {len(trials)} trials", flush=True)\n    meta = pd.DataFrame(metadata)\n    X = np.concatenate(arrays)\n    valid = np.concatenate(masks)\n    phase_valid = np.concatenate(phase_masks)\n    chronology = []\n    for subject in sorted(meta.subject.unique()):\n        train3 = meta[(meta.subject == subject) & (meta.split == "train") & (meta.run == 3)]\n        test3 = meta[(meta.subject == subject) & (meta.split == "test") & (meta.run == 3)]\n        if len(train3) and len(test3):\n            train_report = next(r for r in file_reports if r["subject"] == subject and r["run"] == 3 and r["split"] == "train")\n            test_report = next(r for r in file_reports if r["subject"] == subject and r["run"] == 3 and r["split"] == "test")\n            join_dt = test_report["first_time"] - train_report["last_time"]\n            chronology.append({"subject": subject, "train_trials": len(train3), "test_trials": len(test3),\n                               "all_labeled_cues_before_test": bool(train3.cue_time.max() < test3.cue_time.min()),\n                               "file_join_dt_seconds": join_dt,\n                               "files_abut_one_sample": bool(abs(join_dt / train_report["median_dt_seconds"] - 1) < .01),\n                               "recording_time_ranges_overlap": bool(test_report["first_time"] <= train_report["last_time"]),\n                               "train_internal_discontinuities": len(train_report["discontinuity_rows"]),\n                               "test_internal_discontinuities": len(test_report["discontinuity_rows"]),\n                               "last_train_cue_time": float(train3.cue_time.max()),\n                               "first_test_cue_time": float(test3.cue_time.min()),\n                               "sum_observed_trials": len(train3) + len(test3)})\n    counts = meta.groupby(["split", "subject", "run"], sort=True).agg(\n        trials=("epoch_index", "size"), rest=("label", lambda s: (s == "rest").sum()),\n        move=("label", lambda s: (s == "move").sum())).reset_index()\n    report = {\n        "competition": competition, "cache_version": 1,\n        "channels": list(CHANNELS), "nominal_fs": FS,\n        "amplitude_units": "Unverified; CSV contains numeric raw device units with large DC offsets",\n        "acquisition_reference": "Not established from delivered CSVs",\n        "predictor_columns": list(CHANNELS), "files": file_reports,\n        "counts": counts.to_dict(orient="records"), "run3_chronology": chronology,\n        "exact_duplicate_files_within_competition": duplicate_files,\n        "exact_duplicate_4s_epochs_within_competition": duplicate_epochs,\n        "train_trials": int((meta.split == "train").sum()), "test_trials": int((meta.split == "test").sum()),\n        "trial_lengths": {col: {"min": float(meta[col].min()), "max": float(meta[col].max()),\n                                 "median": float(meta[col].median())}\n                          for col in ["baseline_samples", "native_cue_samples", "baseline_seconds", "cue_seconds"]},\n        "invalid_recorded_samples": int((~valid).sum()),\n        "invalid_phase_samples": int((~phase_valid).sum()),\n        "invalid_baseline_2s_trials": int((~meta.baseline_2s_valid).sum()),\n        "invalid_task_4p8s_trials": int((~meta.task_4p8s_valid).sum()),\n        "invalid_task_5s_trials": int((~meta.task_5s_valid).sum()),\n        "sample_submission_available": False,\n        "test_order_contract": "0-based, lexicographically sorted test filenames then chronological cue order",\n        "run_mixture": meta.groupby(["split", "run"]).size().rename("trials").reset_index().to_dict(orient="records"),\n        "subjects": {split: sorted(meta.loc[meta.split == split, "subject"].unique().tolist())\n                     for split in ["train", "test"]},\n    }\n    output = output_root / competition\n    output.mkdir(parents=True, exist_ok=True)\n    # Uncompressed cache favors repeatable short HPC reads over CPU compression.\n    np.savez(output / "epochs.npz", X=X, y=meta.y.to_numpy(dtype=np.int8),\n             valid_mask=valid, phase_valid_mask=phase_valid, fs=np.array(FS),\n             cue_index=np.array(CUE_INDEX), channels=np.array(CHANNELS), cache_version=np.array(1))\n    meta.to_csv(output / "metadata.csv", index=False)\n    counts.to_csv(output / "counts.csv", index=False)\n    (output / "audit.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\ndef load_cache(artifact_root: str | Path, competition: str):\n    """Load one competition and verify row/label/order alignment."""\n    if competition not in COMPETITIONS:\n        raise ValueError(competition)\n    root = Path(artifact_root) / competition\n    with np.load(root / "epochs.npz", allow_pickle=False) as cache:\n        arrays = {key: cache[key] for key in cache.files}\n    if int(arrays["cache_version"]) != 1 or int(arrays["fs"]) != FS or int(arrays["cue_index"]) != CUE_INDEX:\n        raise ValueError("Unsupported cache contract")\n    if arrays["X"].shape[1:] != (len(CHANNELS), EPOCH_SAMPLES) or arrays["X"].dtype != np.float32:\n        raise ValueError("Invalid epoch shape or dtype")\n    if tuple(arrays["channels"]) != CHANNELS:\n        raise ValueError("Invalid cached channel order")\n    for key in ["valid_mask", "phase_valid_mask"]:\n        if arrays[key].shape != (len(arrays["X"]), EPOCH_SAMPLES) or arrays[key].dtype != bool:\n            raise ValueError(f"Invalid {key}")\n    if (arrays["phase_valid_mask"] & ~arrays["valid_mask"]).any():\n        raise ValueError("Phase validity extends beyond recording validity")\n    metadata = pd.read_csv(root / "metadata.csv", keep_default_na=False)\n    if len(metadata) != len(arrays["X"]) or not np.array_equal(metadata.epoch_index, np.arange(len(metadata))):\n        raise ValueError("Metadata and epoch cache are not aligned")\n    if not (metadata.competition == competition).all() or not np.array_equal(metadata.y, arrays["y"]):\n        raise ValueError("Competition or label mismatch")\n    if not metadata.split.isin(["train", "test"]).all():\n        raise ValueError("Unknown cache split")\n    expected_y = metadata.label.map(LABELS).fillna(-1).to_numpy(dtype=np.int8)\n    if not np.array_equal(expected_y, arrays["y"]) or not (metadata.loc[metadata.split == "test", "y"] == -1).all():\n        raise ValueError("Invalid label encoding")\n    if not metadata.loc[metadata.split == "train", "y"].isin([0, 1]).all():\n        raise ValueError("Unlabeled training rows")\n    test_order = metadata.loc[metadata.split == "test", "test_order"].to_numpy()\n    if not np.array_equal(test_order, np.arange(len(test_order))):\n        raise ValueError("Invalid test ordering")\n    return arrays, metadata\n', 'classical.py': '"""Classical EEG representations. No neural dependencies or test-batch fitting."""\nfrom dataclasses import dataclass, asdict\nimport numpy as np\nfrom scipy import linalg, signal\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import SVC\n\nCHANNELS = (\'Fz\', \'C3\', \'Cz\', \'C4\', \'PO7\', \'Pz\', \'PO8\', \'Oz\')\nBANDS = ((1., 4.), (4., 8.), (8., 13.), (13., 20.), (20., 30.), (30., 45.))\nFS, CUE = 250, 750\n\n\n@dataclass(frozen=True)\nclass Recipe:\n    kind: str\n    window: tuple = (0., 4.8)\n    channels: tuple = tuple(range(8))\n    bands: tuple = BANDS\n    relative: bool = False\n    reference: str = \'native\'\n    erp_mode: str = \'baseline\'\n    classifier: str = \'logistic\'\n\n\n# Prespecified, interpretable contrasts, not an automatic model search.\nRECIPES = {\n    \'erp_early\': Recipe(\'erp\', (0., .8)),\n    \'erp_pre\': Recipe(\'erp\', (0., 2.)),\n    \'erp_late\': Recipe(\'erp\', (2., 4.)),\n    \'erp_early_central\': Recipe(\'erp\', (0., .8), (1, 2, 3)),\n    \'erp_early_posterior\': Recipe(\'erp\', (0., .8), (4, 5, 6, 7)),\n    \'erp_baseline\': Recipe(\'erp\', (-2., 0.)),\n    \'power_pre\': Recipe(\'power\', (0., 2.)),\n    \'power_late\': Recipe(\'power\', (2., 4.)),\n    \'power_full\': Recipe(\'power\'),\n    \'power_baseline\': Recipe(\'power\', (-2., 0.)),\n    \'relpower_full\': Recipe(\'power\', relative=True),\n    \'tangent_full\': Recipe(\'tangent\'),\n    \'tangent_central\': Recipe(\'tangent\', channels=(1, 2, 3)),\n    \'tangent_posterior\': Recipe(\'tangent\', channels=(4, 5, 6, 7)),\n    \'csp_full\': Recipe(\'csp\'),\n    \'paired_cov_full\': Recipe(\'paired\'),\n    # Focused cue-response follow-up: each isolates one mechanism.\n    \'erp_pre_car\': Recipe(\'erp\', (0., 2.), reference=\'car\'),\n    \'erp_pre_centered\': Recipe(\'erp\', (0., 2.), erp_mode=\'centered\'),\n    \'erp_pre_unitnorm\': Recipe(\'erp\', (0., 2.), erp_mode=\'unitnorm\'),\n    \'erp_pre_rbf\': Recipe(\'erp\', (0., 2.), classifier=\'rbf\'),\n    \'erp_pre_central\': Recipe(\'erp\', (0., 2.), (1, 2, 3)),\n    \'erp_pre_posterior\': Recipe(\'erp\', (0., 2.), (4, 5, 6, 7)),\n    \'erp_firstsecond\': Recipe(\'erp\', (0., 1.)),\n    \'erp_secondsecond\': Recipe(\'erp\', (1., 2.)),\n}\n\n\ndef crop(X, window, channels=tuple(range(8))):\n    a, b = [CUE + round(t * FS) for t in window]\n    if a < 0 or b > X.shape[-1] or b <= a:\n        raise ValueError(f\'Invalid crop {window} for {X.shape}\')\n    result = np.asarray(X[:, channels, a:b], dtype=np.float64)\n    if not np.isfinite(result).all():\n        raise ValueError(\'Nonfinite data in required crop; inspect the data audit\')\n    return result\n\n\ndef filtered(x, band):\n    # Filter EACH crop independently, including the baseline negative control.\n    # Neither a neighboring trial nor later experimental phase can leak in.\n    centered = x - x.mean(axis=-1, keepdims=True)\n    sos = signal.butter(4, band, btype=\'bandpass\', fs=FS, output=\'sos\')\n    return signal.sosfiltfilt(sos, centered, axis=-1)\n\n\ndef covariance(x, shrink=.05):\n    x = x - x.mean(axis=-1, keepdims=True)\n    c = x @ x.swapaxes(-1, -2) / max(x.shape[-1] - 1, 1)\n    n = x.shape[-2]\n    scale = np.maximum(np.trace(c, axis1=-2, axis2=-1) / n, 1e-12)\n    return (1. - shrink) * c + shrink * scale[..., None, None] * np.eye(n)\n\n\ndef sym_function(c, function):\n    vals, vecs = np.linalg.eigh(c)\n    vals = np.maximum(vals, 1e-12)\n    return (vecs * function(vals)[..., None, :]) @ vecs.swapaxes(-1, -2)\n\n\ndef vectorize(c):\n    p = c.shape[-1]\n    i, j = np.triu_indices(p)\n    return c[..., i, j] * np.where(i == j, 1., np.sqrt(2.))\n\n\ndef extract(X, recipe):\n    """Return trial-local features/covariances; supervised fitting happens later."""\n    x = crop(X, recipe.window, recipe.channels)\n    if recipe.reference == \'car\':\n        x = x - x.mean(axis=-2, keepdims=True)\n    elif recipe.reference != \'native\':\n        raise ValueError(recipe.reference)\n    if recipe.kind == \'erp\':\n        # Runtime trial-local scaling; no pooled target statistics.\n        baseline = crop(X, (-2., 0.), recipe.channels)\n        if recipe.reference == \'car\':\n            baseline = baseline - baseline.mean(axis=-2, keepdims=True)\n        offset = np.median(baseline, axis=-1, keepdims=True)\n        scale = np.maximum(np.std(baseline, axis=-1, keepdims=True), 1e-6)\n        if recipe.window[1] <= 0:\n            offset = np.median(x, axis=-1, keepdims=True)\n            scale = np.maximum(np.std(x, axis=-1, keepdims=True), 1e-6)\n        x = (x - offset) / scale\n        sos = signal.butter(4, 20., btype=\'lowpass\', fs=FS, output=\'sos\')\n        x = signal.sosfiltfilt(sos, x, axis=-1)\n        if recipe.erp_mode in (\'centered\', \'unitnorm\'):\n            x = x - x.mean(axis=-1, keepdims=True)\n        if recipe.erp_mode == \'unitnorm\':\n            x = x / np.maximum(np.sqrt(np.mean(x*x,axis=(-2,-1),keepdims=True)),1e-6)\n        if recipe.erp_mode not in (\'baseline\',\'centered\',\'unitnorm\'):\n            raise ValueError(recipe.erp_mode)\n        # 40-ms averages retain cue-locked shape with modest dimensionality.\n        n = x.shape[-1] // 10\n        x = x[..., :n * 10].reshape(*x.shape[:-1], n, 10).mean(-1)\n        return np.clip(x, -20., 20.).reshape(len(x), -1).astype(\'float32\')\n    covs = np.stack([covariance(filtered(x, band)) for band in recipe.bands], axis=1)\n    if recipe.kind in (\'tangent\', \'csp\'):\n        return covs.astype(\'float32\')\n    power = np.maximum(np.diagonal(covs, axis1=-2, axis2=-1), 1e-12)\n    if recipe.kind == \'power\' and not recipe.relative:\n        return np.log(power).reshape(len(x), -1).astype(\'float32\')\n    base = crop(X, (-2., 0.), recipe.channels)\n    if recipe.reference == \'car\':\n        base = base - base.mean(axis=-2, keepdims=True)\n    base_cov = np.stack([covariance(filtered(base, band)) for band in recipe.bands], axis=1)\n    if recipe.kind == \'power\':\n        base_power = np.maximum(np.diagonal(base_cov, axis1=-2, axis2=-1), 1e-12)\n        return np.log(power / base_power).reshape(len(x), -1).astype(\'float32\')\n    if recipe.kind == \'paired\':\n        b = sym_function(base_cov, lambda v: v ** -.5)\n        generalized = b @ covs @ b\n        eig = np.linalg.eigvalsh(generalized)\n        ratios = np.log(power / np.maximum(np.diagonal(base_cov, axis1=-2, axis2=-1), 1e-12))\n        return np.concatenate([np.log(np.maximum(eig, 1e-12)).reshape(len(x), -1),\n                               ratios.reshape(len(x), -1)], axis=1).astype(\'float32\')\n    raise ValueError(recipe)\n\n\nclass ClassicalModel:\n    """All covariance references/CSP/scaling fitted within the training fold."""\n    def __init__(self, recipe, C=.1):\n        self.recipe, self.C = recipe, C\n\n    def _represent(self, features):\n        c = np.asarray(features, dtype=np.float64)\n        if self.recipe.kind == \'tangent\':\n            trace = np.maximum(np.trace(c, axis1=-2, axis2=-1), 1e-12)\n            normalized = c / trace[..., None, None]\n            whitened = self.reference_[None] @ normalized @ self.reference_[None]\n            values = vectorize(sym_function(whitened, np.log)).reshape(len(c), -1)\n            return np.concatenate([values, np.log(trace)], axis=1)\n        if self.recipe.kind == \'csp\':\n            trace = np.maximum(np.trace(c, axis1=-2, axis2=-1), 1e-12)\n            c = c / trace[..., None, None]\n            values = np.einsum(\'bik,nbij,bjk->nbk\', self.spatial_, c, self.spatial_)\n            return np.log(np.maximum(values, 1e-12)).reshape(len(c), -1)\n        return c\n\n    def fit(self, features, y, sample_weight=None):\n        if self.recipe.kind in (\'tangent\', \'csp\'):\n            c = np.asarray(features, dtype=np.float64)\n            c = c / np.maximum(np.trace(c, axis1=-2, axis2=-1)[..., None, None], 1e-12)\n            if self.recipe.kind == \'tangent\':\n                reference = np.average(c, axis=0, weights=sample_weight)\n                self.reference_ = sym_function(reference, lambda v: v ** -.5)\n            else:\n                class_cov = []\n                for label in (0, 1):\n                    w = None if sample_weight is None else sample_weight[y == label]\n                    class_cov.append(np.average(c[y == label], axis=0, weights=w))\n                spatial = []\n                for a, b in zip(*class_cov):\n                    _, v = linalg.eigh(a, a + b)\n                    k = min(2, len(a) // 2)\n                    spatial.append(v[:, np.r_[0:k, len(a)-k:len(a)]])\n                self.spatial_ = np.stack(spatial)\n        x = self._represent(features)\n        self.scaler_ = StandardScaler().fit(x, sample_weight=sample_weight)\n        x = self.scaler_.transform(x)\n        if self.recipe.classifier == \'logistic\':\n            self.classifier_ = LogisticRegression(C=self.C, max_iter=1000, solver=\'lbfgs\')\n        elif self.recipe.classifier == \'rbf\':\n            self.classifier_ = SVC(C=self.C, kernel=\'rbf\', gamma=\'scale\', probability=True, random_state=20261003)\n        else:\n            raise ValueError(self.recipe.classifier)\n        self.classifier_.fit(x, y, sample_weight=sample_weight)\n        return self\n\n    def predict_proba(self, features):\n        return self.classifier_.predict_proba(self.scaler_.transform(self._represent(features)))\n\n    def config(self):\n        return {\'recipe\': asdict(self.recipe), \'C\': self.C}\n', 'erp_template.py': '"""Inductive ERP-template covariance classifier, with no neural dependencies.\n\nTemplates, covariance reference and scaler are fitted on source-fold data only.\nOne query trial never changes another query\'s features or predictions.\n"""\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom eeg_comp.classical import covariance, sym_function, vectorize\n\n\nclass ERPTemplateCovariance:\n    def __init__(self, C=.1, shrink=.1):\n        self.C, self.shrink = C, shrink\n\n    def _covariances(self, x):\n        x=np.asarray(x,dtype=np.float64)\n        if x.ndim!=3 or x.shape[1:]!=self.shape_ or not np.isfinite(x).all():\n            raise ValueError(\'Expected finite aligned ERP waveforms\')\n        templates=np.broadcast_to(self.templates_[None],(len(x),)+self.templates_.shape)\n        augmented=np.concatenate([templates,x],axis=1)\n        cov=covariance(augmented,self.shrink)\n        # Remove joint scale; retain relative template/trial coupling.\n        return cov / np.maximum(np.trace(cov,axis1=-2,axis2=-1)[...,None,None],1e-12)\n\n    def _features(self, c):\n        centered=self.reference_[None]@c@self.reference_[None]\n        return vectorize(sym_function(centered,np.log))\n\n    def fit(self,x,y):\n        x,y=np.asarray(x,dtype=np.float64),np.asarray(y)\n        if x.ndim!=3 or len(x)!=len(y) or not np.isfinite(x).all() or set(np.unique(y))!={0,1}:\n            raise ValueError(\'Finite source waveforms and both source classes are required\')\n        self.shape_=x.shape[1:]\n        self.templates_=np.concatenate([x[y==label].mean(axis=0) for label in (0,1)],axis=0)\n        c=self._covariances(x)\n        self.reference_=sym_function(c.mean(axis=0),lambda v:v**-.5)\n        f=self._features(c)\n        self.scaler_=StandardScaler().fit(f)\n        self.classifier_=LogisticRegression(C=self.C,solver=\'lbfgs\',max_iter=1000)\n        self.classifier_.fit(self.scaler_.transform(f),y)\n        return self\n\n    def predict_proba(self,x):\n        return self.classifier_.predict_proba(self.scaler_.transform(self._features(self._covariances(x))))\n', 'submission.py': '"""Validate the official zero-based ID contract before writing a candidate."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nEXPECTED_ROWS = {\'within_subject\':680,\'cross_subject\':360}\n\n\ndef build_submission(predictions, metadata, competition):\n    expected = metadata.loc[metadata.split.eq(\'test\')].sort_values([\'subject\',\'run\',\'trial_index\']).copy()\n    count = EXPECTED_ROWS[competition]\n    if len(expected) != count or not expected.competition.eq(competition).all():\n        raise ValueError(\'Wrong competition or test row count\')\n    if not np.array_equal(expected.test_order,np.arange(count)):\n        raise ValueError(\'Test order differs from participant/session/epoch ordering\')\n    if predictions.epoch_index.duplicated().any() or len(predictions) != count:\n        raise ValueError(\'Duplicate or missing prediction rows\')\n    aligned = predictions.set_index(\'epoch_index\').reindex(expected.epoch_index)\n    for column in [\'competition\',\'subject\',\'run\',\'file\',\'trial_index\',\'test_order\']:\n        if column not in aligned or not np.array_equal(aligned[column].to_numpy(),expected[column].to_numpy()):\n            raise ValueError(f\'Prediction mapping mismatch: {column}\')\n    probs = aligned.p_move.to_numpy(dtype=float)\n    if not np.isfinite(probs).all() or ((probs<0)|(probs>1)).any():\n        raise ValueError(\'Invalid or missing probabilities\')\n    return pd.DataFrame({\'ID\':np.arange(count),\'TARGET\':np.where(probs>=.5,\'move\',\'rest\')})\n\n\ndef save_candidate(prediction_path, metadata_path, competition, category, output):\n    prediction_path,metadata_path,output=map(Path,[prediction_path,metadata_path,output])\n    if output.exists():\n        raise FileExistsError(output)\n    frame=build_submission(pd.read_csv(prediction_path),pd.read_csv(metadata_path),competition)\n    output.parent.mkdir(parents=True,exist_ok=True)\n    frame.to_csv(output,index=False)\n    manifest={\'competition\':competition,\'category\':category,\'rows\':len(frame),\n              \'id_min\':0,\'id_max\':len(frame)-1,\'threshold\':.5,\n              \'label_counts\':frame.TARGET.value_counts().to_dict(),\n              \'prediction_file\':str(prediction_path),\'prediction_sha256\':hashlib.sha256(prediction_path.read_bytes()).hexdigest(),\n              \'metadata_sha256\':hashlib.sha256(metadata_path.read_bytes()).hexdigest(),\n              \'csv_sha256\':hashlib.sha256(output.read_bytes()).hexdigest(),\n              \'status\':\'local candidate; not uploaded; Kaggle execution still required\',\n              \'ordering_source\':\'Official competition evaluation pages; see docs/RULES_AND_SUBMISSION.md\'}\n    output.with_suffix(\'.manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    return manifest\n', '__init__.py': '"""Embedded pure classical ERP-template pipeline."""\n'}
source_root = OUTPUT_DIR / 'embedded_source'
package_root = source_root / 'eeg_comp'
package_root.mkdir(parents=True, exist_ok=True)
for filename, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[filename]:
        raise ValueError('Embedded source checksum mismatch: ' + filename)
    (package_root / filename).write_text(source)
if any(name == 'eeg_comp' or name.startswith('eeg_comp.') for name in sys.modules):
    raise RuntimeError('Restart the kernel before executing this standalone notebook')
sys.path.insert(0, str(source_root.resolve()))
importlib.invalidate_caches()
import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib
from eeg_comp.data import CHANNELS, FS, CUE_INDEX, identify_file, parse_markers, extract_epochs
from eeg_comp.classical import RECIPES, extract
from eeg_comp.erp_template import ERPTemplateCovariance
from eeg_comp.submission import build_submission
random.seed(SEED)
np.random.seed(SEED)
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__}
print(json.dumps({'competition': COMPETITION, 'model': 'ERPTemplateCovariance',
                  'versions': VERSIONS}, indent=2))


## Load and validate raw Cross Subject recordings


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def locate_data_root(explicit_root):
    # Never recursively search /kaggle/input: the other challenge may be attached.
    if explicit_root is not None:
        candidates = [Path(explicit_root)]
    else:
        candidates = [Path('/kaggle/input') / SLUG,
                      Path('/kaggle/input/competitions') / SLUG]
    found = [root.resolve() for root in candidates
             if (root / 'Training set').is_dir() and (root / TEST_FOLDER).is_dir()]
    if len(set(found)) != 1:
        raise FileNotFoundError('Set DATA_ROOT to this challenge folder containing '
                                f'Training set and {TEST_FOLDER!r}; candidates={candidates}')
    root = found[0]
    # Explicit roots must still have the selected challenge's test layout only.
    other_test = ('Cross participant test set' if COMPETITION == 'within_subject'
                  else 'Single subject test set')
    if (root / other_test).exists():
        raise ValueError('Both competition test folders found in one input root')
    return root


def load_raw_competition(root):
    arrays, masks, phase_masks, rows, file_manifest = [], [], [], [], []
    test_order = 0
    for split, folder, expected_files in [('train', 'Training set', 50),
                                          ('test', TEST_FOLDER, EXPECTED_TEST_FILES)]:
        paths = list((root / folder).glob('*.csv'))
        paths.sort(key=lambda path: (*identify_file(path, COMPETITION, split), path.name))
        if len(paths) != expected_files:
            raise ValueError(f'{split}: expected {expected_files} EEG files, found {len(paths)}')
        for path in paths:
            subject, run = identify_file(path, COMPETITION, split)
            frame = pd.read_csv(path)
            trials, marker_report = parse_markers(frame['Marker_val'], split)
            X_part, valid, phase_valid, trial_rows, timing = extract_epochs(frame, trials)
            # Fail closed: no filling, padding, dropping, or substituting unsupported windows.
            for lo, hi in [(-2., 0.), (0., 4.8)]:
                a, b = CUE_INDEX + round(lo * FS), CUE_INDEX + round(hi * FS)
                if not (valid[:, a:b] & phase_valid[:, a:b]).all():
                    raise ValueError(f'{path.name}: unsupported required phase {lo, hi}')
            for row in trial_rows:
                row.update(epoch_index=len(rows), competition=COMPETITION, split=split,
                           subject=subject, run=run, file=path.relative_to(root).as_posix(),
                           test_order=test_order if split == 'test' else -1)
                if split == 'test':
                    test_order += 1
                rows.append(row)
            file_manifest.append({'file': path.relative_to(root).as_posix(), 'split': split,
                                  'subject': subject, 'run': run, 'trials': len(trials),
                                  'sha256': sha256_file(path), **marker_report,
                                  'median_dt_seconds': timing['median_dt_seconds']})
            arrays.append(X_part)
            masks.append(valid)
            phase_masks.append(phase_valid)
            print(f'{COMPETITION}: {path.name}, {len(trials)} trials', flush=True)
    meta = pd.DataFrame(rows)
    X, valid, phase_valid = map(np.concatenate, [arrays, masks, phase_masks])
    train, test = meta[meta.split.eq('train')], meta[meta.split.eq('test')]
    if len(train) != 1795 or len(test) != EXPECTED_TEST_ROWS:
        raise ValueError('Raw trial counts differ from audited competition release')
    if train.subject.nunique() != 17 or test.subject.nunique() != EXPECTED_TEST_PEOPLE:
        raise ValueError('Unexpected participant counts')
    if not set(train.y.unique()) == {0, 1} or not test.y.eq(-1).all():
        raise ValueError('Invalid label visibility')
    if COMPETITION == 'within_subject':
        if set(train.subject) != set(test.subject) or not test.run.eq(3).all():
            raise ValueError('Within-subject participant/run mismatch')
    elif set(train.subject) & set(test.subject):
        raise ValueError('Cross-subject targets overlap training people')
    ordered = test.sort_values(['subject', 'run', 'trial_index'])
    if not np.array_equal(ordered.test_order, np.arange(EXPECTED_TEST_ROWS)):
        raise ValueError('Test order is not participant/session/original epoch order')
    if ordered.duplicated(['file', 'trial_index']).any():
        raise ValueError('Duplicate test trial mapping')
    if not ordered.groupby('file').size().eq(40).all():
        raise ValueError('Expected exactly forty explicitly marked trials per test file')
    return X, valid, phase_valid, meta, file_manifest


data_root = locate_data_root(DATA_ROOT)
X, valid, phase_valid, meta, input_files = load_raw_competition(data_root)
y = meta.y.to_numpy(dtype=np.int8)
meta.to_csv(OUTPUT_DIR / 'metadata.csv', index=False)
print(meta.groupby(['split', 'run']).size())


## Fit the frozen ERP-template model and verify saved-model replay


In [ ]:
if RECIPE != 'erp_pre' or C != .1 or SHRINK != .1:
    raise ValueError('This notebook implements the frozen ERP-template configuration')
recipe = RECIPES[RECIPE]
features = np.concatenate([extract(X[start:start + 128], recipe)
                           for start in range(0, len(X), 128)])
if features.shape != (len(meta), 400) or not np.isfinite(features).all():
    raise ValueError('Expected finite 8-channel by 50-bin ERP features')
waveforms = features.reshape(len(meta), 8, 50)
support = np.flatnonzero(meta.split.eq('train').to_numpy())
query = np.flatnonzero(meta.split.eq('test').to_numpy())
if set(y[support]) != {0, 1} or not np.all(y[query] == -1):
    raise ValueError('Invalid source or target label visibility')
if set(meta.iloc[support].subject) & set(meta.iloc[query].subject):
    raise ValueError('Target participants overlap source participants')
model = ERPTemplateCovariance(C=C, shrink=SHRINK)
model.fit(waveforms[support], y[support])
probability = model.predict_proba(waveforms[query])[:, 1]
test = meta.iloc[query].copy()
test['p_move'] = probability
test = test.sort_values('test_order')
fit_manifest = [{
    'participant': None,
    'training_trials': len(support), 'test_trials': len(query),
    'training_participants': sorted(meta.iloc[support].subject.unique()),
    'training_epoch_indices': support.tolist(),
    'test_epoch_indices': query.tolist(),
    'weighting': 'empirical; no sample weights',
}]
model_path = OUTPUT_DIR / 'erp_template_model.joblib'
joblib.dump({'model': model, 'competition': COMPETITION, 'category': 'ML',
             'build_id': BUILD_ID, 'source_sha256': SOURCE_HASHES,
             'config': {'recipe': RECIPE, 'C': C, 'shrink': SHRINK, 'threshold': .5},
             'training_epoch_indices': support, 'test_epoch_indices': query}, model_path)
reloaded_model = joblib.load(model_path)['model']
replayed = reloaded_model.predict_proba(waveforms[query])[:, 1]
if not np.array_equal(probability, replayed):
    raise ValueError('Immediate saved-model replay probabilities differ')
replay_manifest = {'model_file': model_path.name, 'n': len(query),
                   'probabilities_exactly_equal': True,
                   'labels_identical': bool(np.array_equal(probability >= .5, replayed >= .5)),
                   'max_abs_probability_difference': float(np.max(np.abs(probability-replayed)))}
test['prediction'] = np.where(test.p_move >= .5, 'move', 'rest')
test.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
print(f'Fitted {len(support)} source trials; predicted {len(query)} target trials; saved-model replay exact')


## Export and verify the official submission


In [ ]:
submission = build_submission(test, meta, COMPETITION)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
reloaded = pd.read_csv(OUTPUT_DIR / 'submission.csv')
if list(reloaded.columns) != ['ID', 'TARGET'] or not np.array_equal(reloaded.ID, np.arange(EXPECTED_TEST_ROWS)):
    raise ValueError('Saved submission has invalid columns or IDs')
if not set(reloaded.TARGET).issubset({'rest', 'move'}):
    raise ValueError('Saved submission has invalid labels')
neural_imports = sorted(name for name in sys.modules
                        if name.split('.')[0] in {'torch', 'tensorflow', 'keras', 'jax'})
if neural_imports:
    raise RuntimeError(f'Neural dependencies imported in pure ML execution: {neural_imports[:5]}')
manifest = {
    'competition': COMPETITION, 'competition_slug': SLUG, 'category': 'ML',
    'notebook': NOTEBOOK_NAME, 'build_id': BUILD_ID, 'source_sha256': SOURCE_HASHES,
    'build_provenance': BUILD_PROVENANCE,
    'seed': SEED, 'versions': VERSIONS, 'python_executable': sys.executable,
    'platform': platform.platform(), 'thread_limit': THREADS,
    'config': {'recipe_name': RECIPE, 'recipe': asdict(recipe), 'C': C,
               'covariance_shrinkage': SHRINK, 'weighting': 'empirical; no sample weights',
               'classifier': 'ERPTemplateCovariance'},
    'information_budget': 'source-only class templates, covariance reference, scaler and classifier; trial-local target inference',
    'selection': 'Frozen development-selected ERP-template model; full retraining on all 17 supplied source participants',
    'threshold': .5, 'label_mapping': {'rest': 0, 'move': 1},
    'channels': list(CHANNELS), 'nominal_fs': FS,
    'required_phase_windows': [[-2., 0.], [0., 4.8]],
    'model_phase_windows': [[-2., 0.], [0., 2.]],
    'ordering': 'participant, numeric run, original marker order; zero-based IDs',
    'ordering_source': f'https://www.kaggle.com/competitions/{SLUG}/overview/evaluation',
    'rows': len(submission), 'id_min': 0, 'id_max': len(submission) - 1,
    'label_counts': submission.TARGET.value_counts().to_dict(),
    'raw_data_root': str(data_root), 'raw_inputs': input_files, 'fits': fit_manifest,
    'saved_model_replay': replay_manifest,
    'artifacts_sha256': {name: sha256_file(OUTPUT_DIR / name)
                        for name in ['metadata.csv', 'test_predictions.csv', 'submission.csv',
                                     'erp_template_model.joblib']},
    'elapsed_seconds': time.monotonic() - STARTED,
    'execution_context': ('kaggle' if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') else 'local'),
    'status': 'raw-data training and inference completed; no upload performed',
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(submission.head())
print(f'Saved {len(submission)} rows to {OUTPUT_DIR / "submission.csv"}')
print(f'Elapsed: {manifest["elapsed_seconds"]:.1f} seconds')
